# C1-R — Classical suite under selection protocol v2 (post-hoc for C1)

C1's validation fold (2022, **39 KEV**) did not predict test rank: LightGBM was selected and came 5th on test. Protocol v2 (pre-registered in `reports/analysis_plan_preregistration.md`, commit e21fe21, *before* this run) uses a wider fold: train on CVEs published < 2021-05-01, validate on 2021-05-01 … 2022-12-31 with KEV as of 2023-01-01 — **652 train / 145 validation positives**. The same 20 configs + default per family as C1 are re-scored; the selected configs are refitted with 10 seeds and tested once.

**Status:** C1's primary verdict stands (H2, H1b not supported). C1-R is reported next to it, never instead of it. v2 is binding for C2–C6.

This notebook reads saved outputs. Full run (~5 h on 5 threads): `python -m src.models.c1_revised --stage tune|final|diagnostic|analyse`.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 220)

## Validation search on the v2 fold

In [2]:
t = pd.read_csv("outputs/c1r_tuning.csv")
t.groupby("family").val_pr_auc.describe().round(4)

,count,mean,std,min,25%,50%,75%,max
family,,,,,,,,
catboost,21.0,0.0535,0.0078,0.0392,0.0482,0.0559,0.0602,0.0622
decision_tree,21.0,0.0190,0.0094,0.0036,0.0145,0.0167,0.0192,0.0516
lightgbm,21.0,0.0436,0.0124,0.0260,0.0333,0.0382,0.0542,0.0640
logreg,21.0,0.0806,0.0013,0.0785,0.0795,0.0805,0.0812,0.0827
random_forest,21.0,0.0530,0.0096,0.0377,0.0445,0.0528,0.0603,0.0678
xgboost,21.0,0.0464,0.0136,0.0306,0.0354,0.0434,0.0520,0.0812


In [3]:
pd.read_csv("outputs/c1r_selected_configs.csv", index_col=0)

,config
catboost,"{""iterations"": 255, ""learning_rate"": 0.0925289..."
decision_tree,"{""max_depth"": 11, ""min_samples_leaf"": 56, ""cri..."
lightgbm,"{""n_estimators"": 470, ""learning_rate"": 0.01983..."
logreg,"{""C"": 0.14945139248881545}"
random_forest,"{""max_depth"": 10, ""min_samples_leaf"": 31, ""max..."
xgboost,"{""n_estimators"": 359, ""learning_rate"": 0.01358..."


## Test results (10 seeds, mean ± std) — C1-R vs C1

In [4]:
r1 = pd.read_csv("outputs/c1_results_by_seed.csv"); r1 = r1[r1.variant == "tuned"]
r2 = pd.read_csv("outputs/c1r_results_by_seed.csv")
cmp = pd.concat({"C1 (v1 fold)": r1.groupby("family")[["pr_auc", "precision_at_100"]].mean(),
                 "C1-R (v2 fold)": r2.groupby("family")[["pr_auc", "precision_at_100"]].mean()}, axis=1)
cmp.round(4).sort_values(("C1-R (v2 fold)", "pr_auc"), ascending=False)

C1 (v1 fold)                  C1-R (v2 fold)                 
                    pr_auc precision_at_100         pr_auc precision_at_100
family                                                                     
logreg              0.0188            0.090         0.0235            0.120
xgboost             0.0241            0.120         0.0220            0.132
lightgbm            0.0098            0.044         0.0168            0.111
catboost            0.0191            0.110         0.0146            0.098
random_forest       0.0132            0.060         0.0132            0.060
decision_tree       0.0087            0.060         0.0116            0.065

In [5]:
pd.read_csv("outputs/c1r_summary.csv", index_col=[0, 1])

,,pr_auc,precision_at_10,precision_at_100,ndcg_at_100,mrr,roc_auc,brier,ece
family,variant,,,,,,,,
logreg,tuned,0.0235 ± 0.0,0.2 ± 0.0,0.12 ± 0.0,0.1404 ± 0.0,0.25 ± 0.0,0.8344 ± 0.0,0.0013 ± 0.0,0.0003 ± 0.0
xgboost,tuned,0.022 ± 0.0002,0.16 ± 0.049,0.132 ± 0.006,0.1394 ± 0.0046,0.1291 ± 0.0236,0.8274 ± 0.0023,0.0014 ± 0.0,0.0007 ± 0.0
lightgbm,tuned,0.0168 ± 0.0012,0.13 ± 0.09,0.111 ± 0.0145,0.1153 ± 0.0136,0.1672 ± 0.0785,0.8209 ± 0.0033,0.0014 ± 0.0,0.0001 ± 0.0
catboost,tuned,0.0146 ± 0.0025,0.12 ± 0.0872,0.098 ± 0.0271,0.1026 ± 0.0263,0.1196 ± 0.0266,0.8131 ± 0.0029,0.0014 ± 0.0,0.001 ± 0.0001
random_forest,tuned,0.0132 ± 0.0005,0.3 ± 0.0,0.06 ± 0.01,0.077 ± 0.0084,0.2093 ± 0.0427,0.8251 ± 0.0016,0.0014 ± 0.0,0.0024 ± 0.0001
decision_tree,tuned,0.0116 ± 0.0005,0.23 ± 0.0458,0.065 ± 0.005,0.0897 ± 0.0081,0.3867 ± 0.2304,0.7364 ± 0.0086,0.0014 ± 0.0,0.001 ± 0.0001


## H2 and H1b re-tested under v2 (paired bootstrap, seed-averaged scores)

In [6]:
pd.read_csv("outputs/c1r_hypothesis_tests.csv", index_col=0).round(4)

,pr_auc_diff,pr_auc_ci_low,pr_auc_ci_high,pr_auc_p_not_better,p_at_100_diff,p_at_100_ci_low,p_at_100_ci_high,p_at_100_p_not_better
H2_xgboost_vs_logreg,-0.0013,-0.0070,0.0031,0.744,0.02,-0.02,0.05,0.422
H2_xgboost_vs_decision_tree,0.0095,0.0016,0.0209,0.006,0.07,0.01,0.12,0.018
H1b_logreg_vs_epss_pit,0.0060,0.0013,0.0121,0.008,0.01,-0.03,0.05,0.400
secondary_lightgbm_vs_logreg,-0.0060,-0.0154,0.0004,0.964,-0.01,-0.05,0.03,0.758
secondary_lightgbm_vs_decision_tree,0.0048,-0.0022,0.0132,0.096,0.04,-0.01,0.10,0.100
secondary_catboost_vs_logreg,-0.0078,-0.0185,-0.0027,0.994,0.00,-0.03,0.02,0.772
secondary_catboost_vs_decision_tree,0.0030,-0.0040,0.0109,0.196,0.05,0.00,0.10,0.062


**Reading.** Best-validation GBM is now XGBoost (5th → 2nd-best on test vs C1's pick). H2 is still not supported: XGBoost does not beat LR (−0.0013 [−0.0070, 0.0031]); it does beat the decision tree. H1b: LR beats point-in-time EPSS on PR-AUC (+0.0060 [0.0013, 0.0121]) but not on P@100 (+0.01 [−0.03, 0.05]); H1b requires both, so it is **not supported** as registered — a partial result.

## Selection reliability: does validation rank predict test rank?

All 126 configs fitted once on the full training set (seed 0) and scored on test (diagnostic only — never used for selection). Spearman between validation and test PR-AUC, per fold.

In [7]:
pd.read_csv("outputs/c1r_selection_reliability.csv").round(4)

,family,n_configs,spearman_val_c1_fold,p_val_c1_fold,test_of_pick_val_c1_fold,spearman_val_v2_fold,p_val_v2_fold,test_of_pick_val_v2_fold,test_best_possible,test_median_config
0,catboost,21,0.7351,0.0001,0.0193,0.5662,0.0075,0.0140,0.0198,0.0162
1,decision_tree,21,0.5218,0.0153,0.0098,0.5750,0.0064,0.0123,0.0148,0.0096
2,lightgbm,21,0.1610,0.4856,0.0102,0.6818,0.0007,0.0160,0.0213,0.0117
3,logreg,21,-0.2792,0.2203,0.0188,0.2234,0.3304,0.0235,0.0243,0.0233
4,random_forest,21,0.8244,0.0000,0.0136,0.6489,0.0015,0.0136,0.0167,0.0100
5,xgboost,21,0.4026,0.0704,0.0255,0.7740,0.0000,0.0220,0.0255,0.0127
6,pooled,126,0.6848,0.0000,NaN,0.7774,0.0000,NaN,NaN,NaN


The v2 fold ranks configurations more like the test set does: pooled ρ 0.68 → 0.78, and within the GBM families that failed in C1 the gain is large (LightGBM 0.16 → 0.68, XGBoost 0.40 → 0.77). Within LR the configs barely differ on test (0.0181–0.0243), so its ρ is uninformative. The test PR-AUC of the v2 pick is closer to the best achievable for LightGBM and LR; for XGBoost and CatBoost the v1 pick happened to be better. This supports the v2 fold for C2–C6, with the caveat that ρ is estimated on 21 configs per family.

## Friedman across families (46 monthly batches)

In [8]:
pd.read_csv("outputs/c1r_friedman.csv", index_col=0).round(4)

,statistic,p_value,n_months,rank_xgboost,rank_lightgbm,rank_logreg,rank_catboost,rank_decision_tree,rank_random_forest
precision_at_10,17.9630,0.0030,46.0,3.1739,3.1957,3.2500,3.5109,3.9022,3.9674
ndcg_at_10,19.2684,0.0017,46.0,3.0870,3.2609,3.1957,3.5326,3.9130,4.0109
map,48.0373,0.0000,46.0,2.9783,3.2174,2.7609,3.2391,5.1304,3.6739
mrr,15.0900,0.0100,46.0,3.0543,3.5326,2.8261,3.7717,4.0652,3.7500


In [9]:
pd.read_csv("outputs/c1r_nemenyi_map.csv", index_col=0).round(3)

,catboost,decision_tree,lightgbm,logreg,random_forest,xgboost
family,,,,,,
catboost,1.000,0.000,1.000,0.824,0.876,0.985
decision_tree,0.000,1.000,0.000,0.000,0.003,0.000
lightgbm,1.000,0.000,1.000,0.851,0.851,0.990
logreg,0.824,0.000,0.851,1.000,0.178,0.994
random_forest,0.876,0.003,0.851,0.178,1.000,0.477
xgboost,0.985,0.000,0.990,0.994,0.477,1.000
